# Global Optimization (Differential Evolution)

The following global optimizers are implemented in Optiland:
1. Differential Evolution
2. Dual Annealing
3. SHGO
4. Basin-hopping

These optimizers wrap the scipy.optimize implementations.

In [ ]:
import numpy as np

from optiland import optic, optimization
from optiland.optimization import minimize


Define a starting lens:

In [ ]:
lens = optic.Optic()

# add surfaces
lens.surfaces.add(index=0, radius=np.inf, thickness=np.inf)
lens.surfaces.add(index=1, radius=40, thickness=5, material="SK16", is_stop=True)
lens.surfaces.add(index=2, radius=-100, thickness=50)
lens.surfaces.add(index=3)

# set aperture
lens.set_aperture(aperture_type="EPD", value=20)

# set fields
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)
lens.fields.add(y=5)

# set wavelength
lens.wavelengths.add(value=0.55, is_primary=True)

lens.draw()

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization):

In [ ]:
"""
Add a focal length operand and wavefront error operands for all fields.

Use Gaussian quadrature distribution for the rays (see distribution documentation for
more information).
"""

# focal length target
input_data = {"optic": lens}
problem.add_operand(operand_type="f2", target=60, weight=1, input_data=input_data)

# wavefront error target
for field in lens.fields.get_field_coords():
    input_data = {
        "optic": lens,
        "Hx": field[0],
        "Hy": field[1],
        "num_rays": 3,
        "wavelength": 0.55,
        "distribution": "gaussian_quad",
    }
    problem.add_operand(
        operand_type="OPD_difference",
        target=0,
        weight=1,
        input_data=input_data,
    )

Define variables - let both radii of curvature vary. We will use differential evolution, which requires bounds for all variables.

In [ ]:
problem.add_variable(lens, "radius", surface_number=1, min_val=-500, max_val=500)
problem.add_variable(lens, "radius", surface_number=2, min_val=-500, max_val=500)

Let thicknesses to image plane vary:

In [ ]:
problem.add_variable(lens, "thickness", surface_number=2, min_val=30, max_val=100)

Check initial merit function value and system properties:

In [ ]:
problem.info()

Define optimizer:

Run optimization:

In [ ]:
result = minimize(problem, "differential_evolution", maxiter=256, disp=False, workers=-1)
print(result)


Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
lens.draw(num_rays=5)